# Drug Discovery (SMILES sample demo)

Original step-by-step Colab walkthrough, preserved for reference. The maintained implementation lives in `src/smiles_vae`.

In [ ]:
#Step 1: Install RDKit and Import Libraries
# Install RDKit
# This command installs the RDKit library, which is used for cheminformatics tasks like molecule manipulation.
%pip install -q rdkit tensorflow matplotlib

# Import necessary libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from rdkit import Chem
from rdkit.Chem import Descriptors

#tensorflow and keras for building and training the neural network.
#rdkit for handling chemical structures and calculating molecular descriptors.

In [ ]:
#Step 2: Define a Sample Dataset
# small sample dataset of SMILES strings
sample_smiles = [
    'CCO',  # Ethanol
    'CC(=O)O',  # Acetic acid
    'CCC',  # Propane
    'CCN',  # Ethylamine
    'CCOCC',  # Diethyl ether
]
#SMILES (Simplified Molecular Input Line Entry System)

# Ensure all SMILES strings are treated as strings
smiles = np.array(sample_smiles).astype(str)

In [ ]:
#Step 3: Create Character Set and Mappings
# Define a simple character set for SMILES
char_set = set(''.join(smiles))
char_to_index = {char: i for i, char in enumerate(sorted(char_set))}
index_to_char = {i: char for char, i in char_to_index.items()}

max_len = max(len(smile) for smile in smiles)

In [ ]:
#Step 5: One-Hot Encoding of SMILES Strings (3D array initialized with zeros)
def smiles_to_one_hot(smiles, max_len, char_to_index):
    one_hot = np.zeros((len(smiles), max_len, len(char_to_index)), dtype=np.float32)
    for i, smile in enumerate(smiles):
        for j, char in enumerate(smile):
            if char in char_to_index:
                one_hot[i, j, char_to_index[char]] = 1.0
    return one_hot

one_hot_smiles = smiles_to_one_hot(smiles, max_len, char_to_index)

In [ ]:
#Step 6: Define the Encoder
latent_dim = 2                                                                  #Size of Latent Space
input_shape = (max_len, len(char_to_index))                                     #Shape of Input Data

encoder_inputs = layers.Input(shape=input_shape)
x = layers.Conv1D(64, 3, activation='relu', padding='same')(encoder_inputs)
x = layers.Conv1D(64, 3, activation='relu', padding='same')(x)
x = layers.Flatten()(x)
x = layers.Dense(128, activation='relu')(x)
z_mean = layers.Dense(latent_dim, name='z_mean')(x)                             #Transform input Data into 2D Vector
z_log_var = layers.Dense(latent_dim, name='z_log_var')(x)

In [ ]:
# Step 7: Define the Sampling Layer
# Reparameterization (combining z_mean and z_log_var) ---- To generate Samples
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.random.normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

In [ ]:
#Step 8: Create Encoder Model  (Encoder Layers + Sampling Layers)
z = Sampling()([z_mean, z_log_var])
encoder = models.Model(encoder_inputs, [z_mean, z_log_var, z], name='encoder')

In [ ]:
#Step 9: Define the Decoder
#It takes latent space inputs and reconstructs the original data using dense and transposed convolutional layers.
latent_inputs = layers.Input(shape=(latent_dim,))
x = layers.Dense(128, activation='relu')(latent_inputs)
x = layers.Dense(max_len * 64, activation='relu')(x)
x = layers.Reshape((max_len, 64))(x)
x = layers.Conv1DTranspose(64, 3, activation='relu', padding='same')(x)
x = layers.Conv1DTranspose(64, 3, activation='relu', padding='same')(x)
decoder_outputs = layers.Conv1DTranspose(len(char_to_index), 3, activation='softmax', padding='same')(x)

decoder = models.Model(latent_inputs, decoder_outputs, name='decoder')

In [ ]:
#Step 10: Define the VAE Model (Encoder + Decoder)
# Define the VAE class with an adjustable KL weight
#Adds a KL divergence loss to encourage the latent space to follow a normal distribution.

class VAE(tf.keras.Model):
    def __init__(self, encoder, decoder, kl_weight=1.0, **kwargs):
        super(VAE, self).__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.kl_weight = kl_weight

    def call(self, inputs):
        z_mean, z_log_var, z = self.encoder(inputs)
        reconstructed = self.decoder(z)
        kl_loss = -0.5 * tf.reduce_mean(z_log_var - tf.square(z_mean) - tf.exp(z_log_var) + 1)
        self.add_loss(self.kl_weight * kl_loss)
        return reconstructed

In [ ]:
#Step 11: Compile and Train the Model
vae = VAE(encoder, decoder, kl_weight=0.1)                                      #KL weight = 0.1
vae.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='categorical_crossentropy') #Adam optimizer ; categorical cross-entropy loss

# Train the model and capture the history
history = vae.fit(one_hot_smiles, one_hot_smiles, epochs=25, batch_size=2, validation_split=0.2)

#epoches=25
#batch size=2

In [ ]:
#Step 12: Generate New SMILES Strings
# Function generates new SMILES strings by sampling from the latent space
def generate_smiles(vae, num_samples, max_len, index_to_char):
    z_samples = np.random.normal(size=(num_samples, latent_dim))
    generated_one_hot = vae.decoder.predict(z_samples)
    generated_smiles = []
    for i in range(num_samples):
        chars = [index_to_char[np.argmax(vec)] for vec in generated_one_hot[i]]
        smile = ''.join(chars).strip()
        generated_smiles.append(smile)
    return generated_smiles

#The decoder reconstructs these samples into one-hot encoded SMILES strings, which are then converted back to text.
# Generating new SMILES strings
new_smiles = generate_smiles(vae, 10, max_len, index_to_char)
print(new_smiles)

"""**GIVEN INPUT**
    'CCO',  
    'CC(=O)O',  
    'CCC',  
    'CCN',  
    'CCOCC'

**OBTAINED OUTPUT**
['CCCCNNO', 'CCCCCNO', 'CCC=ONO', 'CCCNNNO', 'CCC=CNO', 'CCCNONO', 'CCC=ONO', 'CCCCNNO', 'CCCCNNO', 'CCCNNNO']

"""

In [ ]:
#Step 13: Analyze Generated SMILES Strings (using RDKit)
for smile in new_smiles:
    mol = Chem.MolFromSmiles(smile)
    if mol:
        mw = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        hbd = Descriptors.NumHDonors(mol)
        hba = Descriptors.NumHAcceptors(mol)
        print(f'SMILES: {smile}, MW: {mw}, LogP: {logp}, HBD: {hbd}, HBA: {hba}')
    else:
        print(f'Invalid SMILES: {smile}')


#It Calculates molecular descriptors such as:-
# 1) Molecular weight (MW)
# 2) LogP  (measure of hydrophobicity)
# 3) Number of Hydrogen bond donors (HBD)
# 4) Number of Hydrogen bond acceptors (HBA)

In [ ]:
#Step 14: Access and Plot Training History
# Access training history
print(history.history.keys())

# Plot training history (example)
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

"""25 epochs:-(1st time)
 ['CC((()=', 'CC(((C=', 'CC((()=', 'CC((()=', 'CC((()=', 'CC(((C=', 'CC((()=', 'CC(((C=', 'CC((()=', 'CC(((C=' ]

25 epochs:-(2nd time)
['CCCCNNO', 'CCCCCNO', 'CCC=ONO', 'CCCNNNO', 'CCC=CNO', 'CCCNONO', 'CCC=ONO', 'CCCCNNO', 'CCCCNNO', 'CCCNNNO']

75 epochs:-
['CCCNCC)', 'CCCNCC)', 'CCCNCC)', 'CCCNCC)', 'CCCNCC)', 'CCCNCC)', 'CCCNCC)', 'CCCNCC)', 'CCCNCC)', 'CCCNCC)']

100 epochs:-(no change)
['CCCNCN)', 'CCCNCN)', 'CCCNCN)', 'CCCNCN)', 'CCCNCN)', 'CCCNCN)', 'CCCNCN)', 'CCCNCN)', 'CCCNCN)', 'CCCNCN)']

250 epochs:- (Overfitting)
['OOOOOOO', 'OOOOOOO', 'OOOOOOO', 'OOOOOOO', 'OOOOOOO', 'OOOOOOO', 'OOOOOOO', 'OOOOOOO', 'OOOOOOO', 'OOOOOOO']

"""